# 01 — Explore JSON Dataset Structure

This notebook explores the structure of the JSON files used in this project.

The goal is to understand:
- The **top-level keys** of the JSON file and what each one represents
- The **units** section, which defines what measurement unit each variable uses
- The **hourly data layout** — how the time-series arrays are organized and how to read them correctly

This is the first step before any analysis or visualization. If we don't understand the structure of the data, we risk misreading values or mixing up variables.

---
## 1. Load the JSON File

We use Python's built-in `json` module to load the file. Once loaded, the JSON becomes a regular Python **dictionary**, so we can access any part of it using standard dictionary syntax like `data["key"]`.

The code below automatically searches the project folder for any `.json` file, so no manual path editing is needed.

In [1]:
import json
import os
import glob

# ── Auto-discover: search current folder and one level up for any .json file ──
search_patterns = [
    "*.json",
    "data/*.json",
    "../data/*.json",
    "../*.json",
]

json_files = []
for pattern in search_patterns:
    json_files.extend(glob.glob(pattern))

if not json_files:
    raise FileNotFoundError(
        "No JSON file found. Place your .json file in the same folder as this notebook "
        "or inside a 'data/' subfolder."
    )

data_path = json_files[0]
print(f"JSON file found: {data_path}")

with open(data_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("File loaded successfully.")
print(f"Data type : {type(data)}")

JSON file found: weather.json
File loaded successfully.
Data type : <class 'dict'>


---
## 2. Explore the Top-Level Keys

Every JSON file has a root level — think of it like the table of contents of the file. Each key at this level points to a different section of the data.

Let's print all the top-level keys first, then explain what each one means.

In [2]:
print("Top-level keys:")
for key in data.keys():
    print(f"  - {key}")

Top-level keys:
  - latitude
  - longitude
  - hourly_units
  - hourly


### What each top-level key typically represents:

| Key | What It Contains |
|-----|------------------|
| `latitude` | The geographic latitude of the location this data was collected for |
| `longitude` | The geographic longitude of the location |
| `generationtime_ms` | How long (in milliseconds) it took the API to generate this response — useful for debugging |
| `utc_offset_seconds` | The UTC offset for the timezone of the location, in seconds |
| `timezone` | The timezone name (e.g., `"Africa/Cairo"`) |
| `timezone_abbreviation` | Short form of the timezone (e.g., `"EET"`) |
| `elevation` | The elevation of the location above sea level, in meters |
| `hourly_units` | A dictionary that maps each variable name to its unit of measurement |
| `hourly` | The actual time-series data — arrays of values recorded once per hour |

> **Note:** Some keys may differ depending on your specific JSON file. If a key above is missing or there are extra keys, update the table to match your data.

In [3]:
scalar_keys = ["latitude", "longitude", "timezone", "timezone_abbreviation",
               "elevation", "utc_offset_seconds"]

print("Location and metadata:")
for key in scalar_keys:
    if key in data:
        print(f"  {key}: {data[key]}")

Location and metadata:
  latitude: 30.0
  longitude: 31.0


---
## 3. Explore the Units

The `hourly_units` section is a dictionary that tells us **what unit each variable is measured in**.

This is critical. For example:
- Is temperature in **°C or °F**?
- Is wind speed in **km/h or m/s**?
- Is precipitation in **mm or inches**?

Without checking units, we might analyze or display data with completely wrong labels — which would make any output misleading.

In [4]:
hourly_units = data.get("hourly_units", {})

print("Hourly units:")
for variable, unit in hourly_units.items():
    print(f"  {variable}: {unit}")

Hourly units:
  time: iso8601
  temperature_2m: °C
  relative_humidity_2m: %


### Why units matter:

Each entry in `hourly_units` maps directly to an array inside `hourly`. So if `hourly_units["temperature_2m"]` says `"°C"`, then every number in `hourly["temperature_2m"]` is in degrees Celsius.

Common variables and their typical units in this type of dataset:

| Variable | Typical Unit | Meaning |
|----------|-------------|--------|
| `time` | ISO 8601 string | Timestamp for each hourly record |
| `temperature_2m` | `°C` | Air temperature 2 meters above ground |
| `relative_humidity_2m` | `%` | Relative humidity percentage |
| `wind_speed_10m` | `km/h` | Wind speed 10 meters above ground |
| `precipitation` | `mm` | Rainfall or snowfall amount |

> Always check `hourly_units` for your specific file — the units above are examples and may differ.

---
## 4. Explore the Hourly Data

The `hourly` section is where the actual measurements live. It is a dictionary where each key is a **variable name**, and each value is a **list (array) of readings** — one per hour.

Let's first see what variables are available inside `hourly`.

In [5]:
hourly = data["hourly"]

print("Hourly keys:")
for key in hourly.keys():
    print(f"  - {key}")

Hourly keys:
  - time
  - temperature_2m
  - relative_humidity_2m


Now let's preview the first few values for `time` and at least two other variables to get a feel for the data.

In [6]:
preview_count = 5

print(f"First {preview_count} values — time:")
print(hourly["time"][:preview_count])

other_keys = [k for k in hourly.keys() if k != "time"]

for key in other_keys[:2]:
    unit = hourly_units.get(key, "?")
    print(f"\nFirst {preview_count} values — {key} ({unit}):")
    print(hourly[key][:preview_count])

First 5 values — time:
['2024-01-01 00:00', '2024-01-01 01:00', '2024-01-01 02:00']

First 5 values — temperature_2m (°C):
[20, 21, 19]

First 5 values — relative_humidity_2m (%):
[60, 65, 70]


---
## 5. Understanding the Hourly Array Layout

This is the most important concept for working with this dataset correctly.

Each variable inside `hourly` is stored as a **separate list**, but all lists are **aligned by index**. This means:

- `hourly["time"][0]` → the timestamp of hour #1
- `hourly["temperature_2m"][0]` → the temperature **during that same hour**
- `hourly["wind_speed_10m"][0]` → the wind speed **during that same hour**

They all share the same index. Index 0 = hour 1, index 1 = hour 2, and so on.

Think of it like columns in a spreadsheet — each column is a separate list, but row 0 always belongs together.

In [7]:
# Each index position refers to the same hourly record across all arrays
print("Aligned index demo (index 0 = first recorded hour):")
print(f"  time             : {hourly['time'][0]}")

other_keys = [k for k in hourly.keys() if k != "time"]
for key in other_keys[:2]:
    unit = hourly_units.get(key, "?")
    print(f"  {key:<25}: {hourly[key][0]} {unit}")

Aligned index demo (index 0 = first recorded hour):
  time             : 2024-01-01 00:00
  temperature_2m           : 20 °C
  relative_humidity_2m     : 60 %


---
## 6. Validate the Hourly Layout — Check Array Lengths

If the dataset is correctly structured, every array inside `hourly` should have **exactly the same length**. If even one array has a different length, something is wrong with the data — it means some values are missing or extra, and index alignment would break.

Let's check this.

In [8]:
lengths = {key: len(hourly[key]) for key in hourly.keys()}

print("Array lengths:")
for key, length in lengths.items():
    print(f"  {key}: {length}")

Array lengths:
  time: 3
  temperature_2m: 3
  relative_humidity_2m: 3


In [9]:
unique_lengths = set(lengths.values())
all_equal = len(unique_lengths) == 1

print(f"All arrays same length: {all_equal}")

if all_equal:
    total_hours = unique_lengths.pop()
    print(f"  → Every array has {total_hours} records "
          f"(≈ {total_hours / 24:.1f} days of hourly data).")
else:
    print("  ⚠️  WARNING: Arrays have different lengths — data may be misaligned!")
    for key, length in lengths.items():
        print(f"     {key}: {length}")

All arrays same length: True
  → Every array has 3 records (≈ 0.1 days of hourly data).


### What this tells us:

If all lengths are equal (✅), we can safely use the index to match values across variables. Every position in any array refers to the same moment in time.

If lengths differ (⚠️), we would need to investigate which variable has missing or extra values before doing any analysis.

---
## 7. (Optional but Recommended) Convert to a Pandas DataFrame

While the raw dictionary format works, it's much easier to read and explore the data as a **DataFrame** — a table where each column is a variable and each row is one hourly record.

This also makes the aligned index concept visually obvious: you can literally see that each row is one timestamp.

In [11]:
import pandas as pd

df = pd.DataFrame(hourly)
df["time"] = pd.to_datetime(df["time"])

print(f"DataFrame shape: {df.shape[0]} rows × {df.shape[1]} columns")
print()
print("First 5 rows (df.head()):")
print(df.head().to_string())

DataFrame shape: 3 rows × 3 columns

First 5 rows (df.head()):
                 time  temperature_2m  relative_humidity_2m
0 2024-01-01 00:00:00              20                    60
1 2024-01-01 01:00:00              21                    65
2 2024-01-01 02:00:00              19                    70


### Why use a DataFrame?

The DataFrame view makes three things immediately clear:

1. **Each row is one hour** — the time column shows exactly which hour it is
2. **Each column is one variable** — all values in a column share the same unit
3. **Row 0 across all columns = the same moment** — the alignment concept is now visually obvious

From here, we can easily filter by date, calculate statistics, plot charts, and more.

---
## Summary

Here is what we learned about the structure of this JSON dataset:

**Top-level keys:**
The JSON file has two types of keys at the root level. Scalar keys like `latitude`, `longitude`, `timezone`, and `elevation` describe the location and settings for the dataset. The two main data keys are `hourly_units` and `hourly`.

**Units (`hourly_units`):**
This section maps every variable name to its unit of measurement. Before reading any number from this dataset, always check `hourly_units` to know what that number actually means — whether it's °C, km/h, mm, %, and so on. Skipping this step can lead to incorrect analysis or misleading outputs.

**Hourly data layout (`hourly`):**
The `hourly` section contains one list per variable. All lists have the same length, and they are **index-aligned** — meaning the same position across all lists always refers to the same point in time. This is the core structure of the dataset, and understanding it is essential for any further work.

**Validation:**
We confirmed that all arrays inside `hourly` have equal length, which means the data is consistently structured and safe to use.

**DataFrame:**
Converting the hourly data to a pandas DataFrame is highly recommended — it makes the structure intuitive to read, and unlocks all of pandas' analysis and visualization tools.

> This notebook is the foundation. All future notebooks in this project will build on this understanding of the data structure.